# Lab 3: Workflow for a receipt batch

A batch of expense claims arrives, each with a photo of its receipt in Amazon S3. In this lab a fixed Workflow reviews them. For every claim, an extractor reads the receipt image and returns typed facts. A policy reviewer and an audit reviewer then check those facts in parallel, and Python joins both reviews into a recommendation. It writes the recommendation to S3 as a decision record.

Your code, not a model, decides which stage runs next and what each stage receives. Models work inside a stage. They never choose the next one, and a claim cannot change the stages. The workflow does not approve a real expense or issue a payment.

## What you will do

1. Run the workflow over the batch and watch each stage in its own pane.
2. Read the evidence: stage timings, typed outputs, what each downstream stage received, and the objects in S3.
3. Run a decision stage with an evidence check on the claim where every stage succeeded but the evidence was not enough.

**Estimated time:** 20-25 minutes.

Run the cells from top to bottom. You do not need to edit any code.

![A fixed workflow reads a receipt image from Amazon S3. The extractor's typed facts fan out to a policy review and an audit review, which run concurrently, and a Python decision joins both before writing a decision record to S3.](images/topology.png)


## Why this is a Workflow

Every claim needs the same stages in the same order: facts first, then both reviews, then the decision. You can write that order down before the batch arrives, so no model has to choose it. A Workflow is that order written as ordinary Python: `await` for a dependency, `asyncio.gather` for stages that can run together, and a function for the decision. The [Strands Workflow guide](https://strandsagents.com/docs/user-guide/sdk/multi-agent/workflow/) describes the pattern this way.

Every Agent stage is another model call, and every stage runs for every claim, including claims that would not need it. Keep arithmetic, validation, and the final decision in plain Python. If one Agent with tools can already answer, or if the route depends on what a stage finds, a Workflow is the wrong choice.


## Setup

Run the `%pip` cell by itself first. If it installs or upgrades anything, restart the kernel. Then select the next cell and choose **Run > Run All Below**. Starting **Run All** above the install cell can stop after a package-triggered kernel restart.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import asyncio
import json
import sys
import time
from decimal import Decimal
from pathlib import Path
from typing import Literal

import boto3
from pydantic import BaseModel, Field
from strands import Agent
from strands.models import BedrockModel

# Workshop helpers shared by every lab, in ../utils.
sys.path.insert(0, str(Path("..").resolve()))
from utils.lab_data import upload_receipts
from utils.live_panes import LivePanes
from utils.stacks import cleanup, find_or_deploy
from utils.validation import write_validation_artifact

The extractor runs on Amazon Nova Pro, which reads the receipt images. The policy and audit reviewers run on Anthropic Claude Sonnet 5. Both models are called in the AWS Region of this notebook's session.

In [ ]:
region = boto3.Session().region_name
nova = BedrockModel(model_id="us.amazon.nova-pro-v1:0", region_name=region)
sonnet = BedrockModel(model_id="global.anthropic.claude-sonnet-5", region_name=region)

### Lab resources and data

The lab stack holds one S3 bucket. The workflow reads receipts from `receipts/` and writes decision records to `decisions/`. At an event the stack already exists. On your own account, the next cell deploys it, which takes about a minute.

`upload_receipts()` then uploads the four synthetic receipt images in `data/receipts/` and prints one line for each. A receipt that is already in the bucket with the same bytes is skipped, so you can rerun the cell safely. Open the images in the file browser if you want to see what the extractor will read.

In [ ]:
outputs = find_or_deploy("workflow", "infra/workflow.yaml")
bucket = outputs["ReceiptBucketName"]
receipt_keys = upload_receipts(bucket, region)

## The claims and the typed handoffs

Each claim names its receipt image, the total the employee claimed, the policy clause that applies, and the prior-claim history. The Agents interpret the image and the short texts. Python owns the stage order, the exact total comparison, the join, and the recommendation.

Two Pydantic models are the handoffs. `ReceiptFacts` is what the extractor returns. It allows `None` for the date and the total, because a real receipt can be torn or stained. `StageReview` is what each reviewer returns. Strands validates each Agent's answer against its model through `structured_output_model`, so the next stage receives checked data instead of prose.

`timed()` records when a stage starts and stops, and whether it completed. It is timing code only: it awaits the stage you give it.

In [ ]:
# The S3 client for receipt images and decision records.
s3 = boto3.client("s3", region_name=region)

# The claim batch: receipt, claimed total, and reviewer evidence.
CLAIMS = {
    "expense-1042": {
        "receipt_key": "receipts/r-1042.png",
        "claimed_total": "92.00",
        "policy_text": (
            "Meals above USD 75 require an exception approved by a Finance Director before the expense date. "
            "The supplied exception was approved by a manager on 2026-09-12."
        ),
        "history_text": "Prior claim expense-0998: Cedar Cafe | 2026-09-10 | USD 92.00 | team dinner",
    },
    "expense-1043": {
        "receipt_key": "receipts/r-1043.png",
        "claimed_total": "38.50",
        "policy_text": "Meals up to and including USD 75 can be submitted without an exception.",
        "history_text": "No prior claims from this merchant.",
    },
    "expense-1044": {
        "receipt_key": "receipts/r-1044.png",
        "claimed_total": "46.20",
        "policy_text": "Taxi rides on business travel are reimbursable with a receipt.",
        "history_text": "No prior claims from this merchant.",
    },
    "expense-1045": {
        "receipt_key": "receipts/r-1045.png",
        "claimed_total": "27.80",
        "policy_text": "Meals up to and including USD 75 can be submitted without an exception.",
        "history_text": "No prior claims from this merchant.",
    },
}


# The typed handoffs between stages.
class ReceiptFacts(BaseModel):
    """What the receipt image shows. None means the receipt does not show it legibly."""

    merchant: str
    expense_date: str | None = Field(description="Printed date as YYYY-MM-DD, or null if missing or illegible")
    currency: str = Field(description="ISO currency code, for example USD")
    total: Decimal | None = Field(gt=0, description="Printed total, or null if missing or illegible")


class StageReview(BaseModel):
    """One reviewer's verdict on the receipt facts."""

    review_type: Literal["policy", "audit"]
    status: Literal["CLEAR", "FLAG"]
    reason: str = Field(description="One short sentence explaining the status")


# The extraction prompt and the two review rules.
EXTRACT_PROMPT = (
    "You read one receipt image. Copy only what is printed: the merchant name, the date as YYYY-MM-DD, the "
    "currency code, and the total. If a field is missing, covered, or illegible, return null for it. "
    "Never guess or infer a value."
)
POLICY_RULE = "Apply only the supplied policy text. FLAG a conflict or a missing required approval; otherwise CLEAR."
AUDIT_RULE = (
    "Compare the receipt facts with the supplied prior-claim history. FLAG an apparent duplicate (same merchant, "
    "date, and total); otherwise CLEAR. Do not apply expense policy."
)


# A helper that records each stage's interval for the timeline.
async def timed(claim_id, stage, work, timeline):
    """Await one stage and record its interval and status, even when it fails."""
    # work is a coroutine that has not started yet, so this start time is taken just before the stage runs.
    entry = {
        "claim": claim_id,
        "stage": stage,
        "status": "failed",  # stays "failed" unless the await below returns
        "started": time.perf_counter(),
    }
    try:
        result = await work
        entry["status"] = "completed"
        return result
    finally:
        # finally runs on success and on error, so a failed stage still lands on the timeline.
        entry["stopped"] = time.perf_counter()
        timeline.append(entry)

## What to look for

Look at the claims and the four receipts before you run the workflow.

Look for:

- Which claims come back `BLOCKED`, and which stage blocks each one?
- Does the audit of `expense-1043` run, although its history is empty?
- What does the workflow recommend for `expense-1045`, whose date is under a stain?
- For one claim, which stages overlap in time?


## Run the workflow

Read the cell top to bottom as the topology. `extract` and `review` are the two kinds of Agent stage. Each creates its Agent and loops over `stream_async`, passing every event to `panes.show(event, name=...)`, which puts it in that stage's pane. The last event holds the `AgentResult`, whose `structured_output` is the validated Pydantic object. `extract` first reads the image bytes from S3 and sends them to Nova Pro as an image content block next to the text.

In `review_claim`, the first `await` is the dependency: nothing downstream starts without typed receipt facts. The two `review` calls inside `asyncio.gather` start together with different inputs, and `gather` waits for both, which is the complete join. The join check rejects a missing or mislabelled review. Only then does `decide`, a plain Python function passed in as a parameter, compare the totals and set the recommendation. Last, `review_claim` writes the decision record to S3. The loop at the bottom runs the claims one after another.

Every Agent sets `callback_handler=None`: `stream_async` still calls each Agent's default handler, which would print the same text a second time below the panes.

**What to look for:** one row of panes per claim. The extract pane finishes before either review pane starts, and the two review panes stream at the same time.

In [ ]:
# The agent stages: extraction, then the two reviews.
async def extract(claim_id, claim, panes):
    """Stage 1: Nova Pro reads the receipt image from S3 and returns typed facts."""
    # The with block closes the S3 download stream once the bytes are read.
    with s3.get_object(Bucket=bucket, Key=claim["receipt_key"])["Body"] as body:
        image = body.read()
    extractor = Agent(
        name="extractor",
        model=nova,
        system_prompt=EXTRACT_PROMPT,
        callback_handler=None,  # no console printing; LivePanes shows the stream instead
    )
    # A multimodal prompt: a list of content blocks, one text block and one image block.
    prompt = [
        {"text": f"Extract the facts printed on receipt {claim['receipt_key']}."},
        {
            "image": {
                "format": "png",
                "source": {"bytes": image},  # raw bytes; boto3 encodes them for the Bedrock request
            },
        },
    ]
    # structured_output_model gives the model a tool shaped like ReceiptFacts and validates its answer.
    async for event in extractor.stream_async(prompt, structured_output_model=ReceiptFacts):
        panes.show(event, name=f"{claim_id} extract")
    # The last streamed event carries the AgentResult; structured_output is the validated ReceiptFacts.
    return event["result"].structured_output


async def review(claim_id, stage, rule, payload, panes):
    """Stages 2a and 2b: a reviewer checks the facts against only the evidence in its payload."""
    reviewer = Agent(
        name=f"{stage}_reviewer",
        model=sonnet,
        system_prompt=(
            f"You are the {stage} reviewer for expense claims. Review only the receipt facts and the evidence "
            f"in the request. Return review_type '{stage}'. {rule}"
        ),
        callback_handler=None,
    )
    # The payload goes in as a JSON string, so the reviewer sees only these fields.
    async for event in reviewer.stream_async(json.dumps(payload), structured_output_model=StageReview):
        panes.show(event, name=f"{claim_id} {stage}")
    return event["result"].structured_output


# The decision stage, in plain Python.
def decide(claim, facts, policy, audit):
    """Stage 3, plain Python: the exact total comparison and the final state."""
    blockers = [f"{review.review_type}: {review.reason}" for review in (policy, audit) if review.status == "FLAG"]
    # Decimal compares exactly, so 92.00 and 92.0 are equal and no float rounding gets in the way.
    if facts.total != Decimal(claim["claimed_total"]):
        blockers.append(f"claimed total {claim['claimed_total']} differs from the receipt total {facts.total}")
    return {
        "recommendation": "BLOCKED" if blockers else "READY_FOR_MANAGER_REVIEW",
        "blockers": blockers,
    }


# The fixed workflow for one claim, from receipt to decision record.
# decide is a parameter so a later cell can rerun the same workflow with a different decision stage.
async def review_claim(claim_id, claim, panes, timeline, run, decide=decide):
    """The fixed workflow for one claim: extract -> (policy + audit) -> join -> decide -> S3."""
    async with asyncio.timeout(300):  # one bound for every model call of this claim
        # Dependency: nothing downstream starts without typed receipt facts.
        facts = await timed(claim_id, "extract", extract(claim_id, claim, panes), timeline)

        # Fan-out: both reviews depend on the facts, not on each other.
        receipt = facts.model_dump(mode="json")  # mode="json" turns the Decimal total into a string
        inputs = {
            "policy": {
                "claim_id": claim_id,
                "receipt": receipt,
                "policy_text": claim["policy_text"],
            },
            "audit": {
                "claim_id": claim_id,
                "receipt": receipt,
                "history_text": claim["history_text"],
            },
        }
        # gather runs both reviews as concurrent tasks and returns their results in argument order.
        policy, audit = await asyncio.gather(
            timed(
                claim_id,
                "policy",
                review(claim_id, "policy", POLICY_RULE, inputs["policy"], panes),
                timeline,
            ),
            timed(
                claim_id,
                "audit",
                review(claim_id, "audit", AUDIT_RULE, inputs["audit"], panes),
                timeline,
            ),
        )

    # Complete join: both mandatory reviews, each with the right label.
    if (policy.review_type, audit.review_type) != ("policy", "audit"):
        raise ValueError(f"{claim_id}: the join needs a policy and an audit review")

    # The decision record, written back to S3. **decide(...) adds the recommendation and blockers keys.
    record = {
        "claim_id": claim_id,
        "run": run,
        **decide(claim, facts, policy, audit),
        "receipt": receipt,
    }
    record["reviews"] = {
        "policy": policy.model_dump(),
        "audit": audit.model_dump(),
    }
    key = f"decisions/{run}/{claim_id}.json"  # one object per claim, grouped by run
    s3.put_object(
        Bucket=bucket,
        Key=key,
        Body=json.dumps(record, indent=2),
        ContentType="application/json",
    )
    # The caller also gets the S3 key and each stage's inputs and typed outputs, which the record does not store.
    return record | {
        "key": key,
        "inputs": inputs,
        "outputs": {
            "extract": facts,
            "policy": policy,
            "audit": audit,
        },
    }


# Run the batch, one claim after another.
panes = LivePanes()
timeline, decisions = [], {}
for claim_id, claim in CLAIMS.items():
    decisions[claim_id] = await review_claim(claim_id, claim, panes, timeline, run="batch")

## Inspect the evidence

The timings show the dependency and the overlap for each claim. The inputs show what each review received besides the receipt facts: the policy reviewer saw only the policy text, and the audit reviewer only the history. Neither saw the other's evidence or conversation. The listing shows the receipts you uploaded and one decision record per claim.

Compare this output with the questions under **What to look for**.

In [ ]:
# When each stage ran, relative to the claim's extraction.
print("stage intervals, in seconds from the start of each claim")
for claim_id in CLAIMS:
    # Index this claim's timeline entries by stage name: extract, policy, audit.
    stages = {entry["stage"]: entry for entry in timeline if entry["claim"] == claim_id}
    origin = stages["extract"]["started"]
    spans = "  ".join(
        f"{name} {entry['started'] - origin:5.2f}-{entry['stopped'] - origin:5.2f}" for name, entry in stages.items()
    )
    print(f"  {claim_id}: {spans}")

# What each stage received and returned.
for claim_id, decision in decisions.items():
    print(f"\n{claim_id}: {decision['recommendation']}")
    for blocker in decision["blockers"]:
        print(f"  blocker  : {blocker}")
    print(f"  extract  : {decision['outputs']['extract'].model_dump(mode='json')}")
    for stage in ("policy", "audit"):
        # Keep only the evidence fields (policy_text or history_text) that this reviewer received.
        evidence = {key: value for key, value in decision["inputs"][stage].items() if key.endswith("_text")}
        output = decision["outputs"][stage]
        print(f"  {stage:<9}: received receipt facts + {evidence}")
        print(f"  {'':<9}  returned {output.status}: {output.reason}")

# The decision records in the bucket.
print(f"\nobjects in s3://{bucket}")
# The paginator follows continuation tokens; search applies a JMESPath expression to every page.
objects = s3.get_paginator("list_objects_v2").paginate(Bucket=bucket)
keys = []
for item in objects.search("Contents[]"):
    keys.append(item["Key"])
    print(f"  {item['Key']:<40} {item['Size']:>7,} bytes")

# Check: every stage completed, the reviews overlapped, records exist.
assert all(entry["status"] == "completed" for entry in timeline), "a stage failed"
for claim_id in CLAIMS:
    stages = {entry["stage"]: entry for entry in timeline if entry["claim"] == claim_id}
    # Dependency: extraction stopped before either review started.
    assert stages["extract"]["stopped"] <= min(stages[name]["started"] for name in ("policy", "audit")), claim_id
    # Overlap: the later review started before the earlier review stopped.
    assert max(stages[name]["started"] for name in ("policy", "audit")) < min(
        stages[name]["stopped"] for name in ("policy", "audit")
    ), f"{claim_id}: the two reviews did not overlap"
    assert f"decisions/batch/{claim_id}.json" in keys, f"no decision record for {claim_id}"

## When every stage succeeds, but the evidence is not enough

Look at `expense-1045` in the batch output. The stain covers the date, and the extractor returned `expense_date: None`, as its prompt asks. Every stage completed with a valid output, and the workflow still made a confident recommendation. If both reviewers returned `CLEAR`, the claim went to a manager as if nothing were missing. If a reviewer noticed the gap and returned `FLAG`, the claim was blocked on a rule the policy text does not contain. Either way the outcome depends on a model's judgment call, and neither says what is true: without a date, nobody can check the submission window or a duplicate on another date.

Structural success is not evidence. Each handoff was valid on its own, and no stage owned the question of whether the facts together were enough for a decision. That check is an output invariant, and it belongs in `decide`, the one stage that makes the decision.

Look for:

- Which fields does the invariant check?
- Does the invariant change the recommendation for any other claim in the batch?

The next cell defines a decision stage with the invariant. It applies the new stage to the batch run's typed outputs, then reruns `expense-1045` with it through the `decide` parameter of `review_claim`.

In [ ]:
# The decision stage, now with an evidence check.
def decide_with_evidence_check(claim, facts, policy, audit):
    """Stage 3, plain Python, now with an output invariant: a decision needs the date and the total."""
    missing = []
    if facts.expense_date is None:
        missing.append("expense_date")
    if facts.total is None:
        missing.append("total")
    if missing:
        return {
            "recommendation": "INSUFFICIENT_EVIDENCE",
            "blockers": [f"receipt does not show: {', '.join(missing)}"],
        }
    blockers = [f"{review.review_type}: {review.reason}" for review in (policy, audit) if review.status == "FLAG"]
    if facts.total != Decimal(claim["claimed_total"]):
        blockers.append(f"claimed total {claim['claimed_total']} differs from the receipt total {facts.total}")
    return {
        "recommendation": "BLOCKED" if blockers else "READY_FOR_MANAGER_REVIEW",
        "blockers": blockers,
    }


# The new decision stage on the typed outputs of the batch run, without another model call.
redecided = {}
for claim_id, workflow_result in decisions.items():
    outputs = workflow_result["outputs"]
    facts = outputs["extract"]
    policy_review = outputs["policy"]
    audit_review = outputs["audit"]
    checked_decision = decide_with_evidence_check(
        CLAIMS[claim_id],
        facts,
        policy_review,
        audit_review,
    )
    redecided[claim_id] = checked_decision["recommendation"]

# Rerun the stained receipt through the workflow with the new stage.
panes = LivePanes()
evidence_timeline = []
checked = await review_claim(
    "expense-1045",
    CLAIMS["expense-1045"],
    panes,
    evidence_timeline,
    run="evidence-check",  # writes to decisions/evidence-check/, next to the batch records
    decide=decide_with_evidence_check,  # the only change: the decision stage
)

### Verify

Every stage completed again, and this time the workflow says what it could not decide. The cell also applied the new decision stage to the typed outputs the batch run already produced, without another model call. The invariant changes only the claim with missing evidence.

The invariant sits at the decision, so both reviews still ran for a claim that could not be decided. Checking right after the extractor would save those two model calls. Checking in `decide` keeps a single place that owns the decision.

In [ ]:
print(f"stages         : {[(entry['stage'], entry['status']) for entry in evidence_timeline]}")
print(f"recommendation : {checked['recommendation']}")
print(f"blockers       : {checked['blockers']}")

print("\nbatch run recommendations, before and after the invariant")
for claim_id, decision in decisions.items():
    print(f"  {claim_id}: {decision['recommendation']:<25} -> {redecided[claim_id]}")

assert all(entry["status"] == "completed" for entry in evidence_timeline), "a stage failed"
assert checked["recommendation"] == "INSUFFICIENT_EVIDENCE", (
    f"got {checked['recommendation']}; the extractor returned {checked['receipt']}"
)
changed = [claim_id for claim_id in decisions if redecided[claim_id] != decisions[claim_id]["recommendation"]]
assert changed == ["expense-1045"], f"the invariant changed {changed}"

## Clean up

The next cell deletes the lab stack only if this notebook deployed it (its `ProvisionedBy` output is `notebook`). It first empties the bucket, the receipts and the decision records, because CloudFormation cannot delete a bucket that holds objects. At an event, Workshop Studio owns the stack and the cell leaves the bucket and its contents in place.

In [ ]:
cleanup("workflow", empty_buckets=[bucket])

In [ ]:
write_validation_artifact("workflow", {
    "claims": sorted(decisions),
    "completed_stages": sum(entry["status"] == "completed" for entry in timeline),
    "invariant_changed": changed,
})

## Takeaways

The topology of a Workflow is application code you can read top to bottom. `await` is a dependency, `asyncio.gather` runs independent stages together and is also the join, and every stage runs for every claim. Models work inside stages and never choose the next one.

Typed handoffs make each boundary between a model and your code explicit. A stage receives only the payload your code builds, and Strands validates each Agent's answer before the next stage reads it.

A valid output from every stage does not make a decision sound. Put the invariants that decide whether the evidence is enough in the Python stage that owns the decision, and give it an outcome such as `INSUFFICIENT_EVIDENCE` that is not a guess.

Amazon Bedrock charged for the model calls, three per claim, and Amazon S3 for the stored objects and requests.

## Where to go next

The [Workflow guide](https://strandsagents.com/docs/user-guide/sdk/multi-agent/workflow/) and the [multi-agent patterns comparison](https://strandsagents.com/docs/user-guide/sdk/multi-agent/multi-agent-patterns/) describe the pattern in more detail.